<a href="https://colab.research.google.com/github/MiguelDz89/MIAR_Algoritmos_Optimizacion/blob/main/Algoritmos_Miguel_Diaz_AG3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Reto 3 ***Algoritmos de Optimización***

Nombre: **Gustavo Roger Bravo Esquivias**

Github Path: https://github.com/gussbrav/Reto3


## Carga de librerias

In [1]:
!pip install requests    # Se realizan llamadas HTTP a páginas de la red


In [27]:
!pip install tsplib95    # Se manejan instancias del problema TSP

Carga de los datos del problema

In [3]:
import urllib.request # Se realizan llamadas HTTP a páginas de la red
import tsplib95       # Se utiliza un módulo para manejar instancias del problema del TSP
import math           # Se usa un módulo de funciones matemáticas para operaciones exponenciales
import random         # Se generan valores aleatorios


#http://elib.zib.de/pub/mp-testdata/tsp/tsplib/

# Se descarga el fichero de datos (matriz de distancias)
data_file = "swiss42.tsp" ;
urllib.request.urlretrieve("http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/swiss42.tsp.gz", data_file + '.gz')
!gzip -d swiss42.tsp.gz     # Se descomprime el fichero de datos



In [4]:
#Carga de datos y generación de objetos
###############################################################################
problem = tsplib95.load(data_file)

#Nodos
Nodos = list(problem.get_nodes())

#Aristas
Aristas = list(problem.get_edges())



In [5]:
Aristas

[(0, 0),
 (0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (0, 5),
 (0, 6),
 (0, 7),
 (0, 8),
 (0, 9),
 (0, 10),
 (0, 11),
 (0, 12),
 (0, 13),
 (0, 14),
 (0, 15),
 (0, 16),
 (0, 17),
 (0, 18),
 (0, 19),
 (0, 20),
 (0, 21),
 (0, 22),
 (0, 23),
 (0, 24),
 (0, 25),
 (0, 26),
 (0, 27),
 (0, 28),
 (0, 29),
 (0, 30),
 (0, 31),
 (0, 32),
 (0, 33),
 (0, 34),
 (0, 35),
 (0, 36),
 (0, 37),
 (0, 38),
 (0, 39),
 (0, 40),
 (0, 41),
 (1, 0),
 (1, 1),
 (1, 2),
 (1, 3),
 (1, 4),
 (1, 5),
 (1, 6),
 (1, 7),
 (1, 8),
 (1, 9),
 (1, 10),
 (1, 11),
 (1, 12),
 (1, 13),
 (1, 14),
 (1, 15),
 (1, 16),
 (1, 17),
 (1, 18),
 (1, 19),
 (1, 20),
 (1, 21),
 (1, 22),
 (1, 23),
 (1, 24),
 (1, 25),
 (1, 26),
 (1, 27),
 (1, 28),
 (1, 29),
 (1, 30),
 (1, 31),
 (1, 32),
 (1, 33),
 (1, 34),
 (1, 35),
 (1, 36),
 (1, 37),
 (1, 38),
 (1, 39),
 (1, 40),
 (1, 41),
 (2, 0),
 (2, 1),
 (2, 2),
 (2, 3),
 (2, 4),
 (2, 5),
 (2, 6),
 (2, 7),
 (2, 8),
 (2, 9),
 (2, 10),
 (2, 11),
 (2, 12),
 (2, 13),
 (2, 14),
 (2, 15),
 (2, 16),
 (2, 17),
 (2, 18),


In [6]:
#Probamos algunas funciones del objeto problem

#Distancia entre nodos
problem.get_weight(0, 1)



15

## Funciones basicas

In [7]:

#Funciones basicas
###############################################################################

#Se genera una solucion aleatoria con comienzo en en el nodo 0
def crear_solucion(Nodos):
  solucion = [Nodos[0]]
  for n in Nodos[1:]:
    solucion = solucion + [random.choice(list(set(Nodos) - set({Nodos[0]}) - set(solucion)))]
  return solucion

#Devuelve la distancia entre dos nodos
def distancia(a,b, problem):
  return problem.get_weight(a,b)

#Devuelve la distancia total de una trayectoria/solucion
def distancia_total(solucion, problem):
  distancia_total = 0
  for i in range(len(solucion)-1):
    distancia_total += distancia(solucion[i] ,solucion[i+1] ,  problem)
  return distancia_total + distancia(solucion[len(solucion)-1] ,solucion[0], problem)

sol_temporal = crear_solucion(Nodos)

distancia_total(sol_temporal, problem), sol_temporal

(3963,
 [0,
  24,
  29,
  28,
  23,
  35,
  17,
  33,
  15,
  37,
  20,
  38,
  22,
  4,
  21,
  1,
  31,
  26,
  25,
  41,
  2,
  11,
  30,
  34,
  36,
  7,
  19,
  27,
  10,
  16,
  18,
  40,
  39,
  9,
  12,
  6,
  3,
  14,
  13,
  5,
  8,
  32])

# BUSQUEDA ALEATORIA

In [8]:
###############################################################################
# BUSQUEDA ALEATORIA
###############################################################################

def busqueda_aleatoria(problem, N):
  #N es el numero de iteraciones
  Nodos = list(problem.get_nodes())

  mejor_solucion = []
  #mejor_distancia = 10e100                         #Inicializamos con un valor alto
  mejor_distancia = float('inf')                    #Inicializamos con un valor alto

  for i in range(N):                                #Criterio de parada: repetir N veces pero podemos incluir otros
    solucion = crear_solucion(Nodos)                #Genera una solucion aleatoria
    distancia = distancia_total(solucion, problem)  #Calcula el valor objetivo(distancia total)

    if distancia < mejor_distancia:                 #Compara con la mejor obtenida hasta ahora
      mejor_solucion = solucion
      mejor_distancia = distancia


  print("Mejor solución:" , mejor_solucion)
  print("Distancia     :" , mejor_distancia)
  return mejor_solucion


#Busqueda aleatoria con 5000 iteraciones
solucion = busqueda_aleatoria(problem, 5000)

Mejor solución: [0, 14, 8, 24, 23, 32, 34, 27, 5, 7, 37, 33, 2, 11, 18, 36, 20, 19, 13, 17, 15, 3, 12, 6, 30, 22, 40, 25, 4, 39, 21, 41, 28, 38, 9, 10, 16, 31, 35, 29, 26, 1]
Distancia     : 3742


# BUSQUEDA LOCAL

In [9]:
###############################################################################
# BUSQUEDA LOCAL
###############################################################################
def genera_vecina(solucion):
  #Generador de soluciones vecinas: 2-opt (intercambiar 2 nodos) Si hay N nodos se generan (N-1)x(N-2)/2 soluciones
  #Se puede modificar para aplicar otros generadores distintos que 2-opt
  #print(solucion)
  mejor_solucion = []
  mejor_distancia = 10e100
  for i in range(1,len(solucion)-1):          #Recorremos todos los nodos en bucle doble para evaluar todos los intercambios 2-opt
    for j in range(i+1, len(solucion)):

      #Se genera una nueva solución intercambiando los dos nodos i,j:
      #  (usamos el operador + que para listas en python las concatena) : ej.: [1,2] + [3] = [1,2,3]
      vecina = solucion[:i] + [solucion[j]] + solucion[i+1:j] + [solucion[i]] + solucion[j+1:]

      #Se evalua la nueva solución ...
      distancia_vecina = distancia_total(vecina, problem)

      #... para guardarla si mejora las anteriores
      if distancia_vecina <= mejor_distancia:
        mejor_distancia = distancia_vecina
        mejor_solucion = vecina
  return mejor_solucion


#solucion = [1, 47, 13, 41, 40, 19, 42, 44, 37, 5, 22, 28, 3, 2, 29, 21, 50, 34, 30, 9, 16, 11, 38, 49, 10, 39, 33, 45, 15, 24, 43, 26, 31, 36, 35, 20, 8, 7, 23, 48, 27, 12, 17, 4, 18, 25, 14, 6, 51, 46, 32]
print("Distancia Solucion Incial:" , distancia_total(solucion, problem))


nueva_solucion = genera_vecina(solucion)
print("Distancia Mejor Solucion Local:", distancia_total(nueva_solucion, problem))


Distancia Solucion Incial: 3742
Distancia Mejor Solucion Local: 3568


In [10]:
#Busqueda Local:
#  - Sobre el operador de vecindad 2-opt(funcion genera_vecina)
#  - Sin criterio de parada, se para cuando no es posible mejorar.
def busqueda_local(problem):
  mejor_solucion = []

  #Generar una solucion inicial de referencia(aleatoria)
  solucion_referencia = crear_solucion(Nodos)
  mejor_distancia = distancia_total(solucion_referencia, problem)

  iteracion=0             #Un contador para saber las iteraciones que hacemos
  while(1):
    iteracion +=1         #Incrementamos el contador
    #print('#',iteracion)

    #Obtenemos la mejor vecina ...
    vecina = genera_vecina(solucion_referencia)

    #... y la evaluamos para ver si mejoramos respecto a lo encontrado hasta el momento
    distancia_vecina = distancia_total(vecina, problem)

    #Si no mejoramos hay que terminar. Hemos llegado a un minimo local(según nuestro operador de vencindad 2-opt)
    if distancia_vecina < mejor_distancia:
      #mejor_solucion = copy.deepcopy(vecina)   #Con copia profunda. Las copias en python son por referencia
      mejor_solucion = vecina                   #Guarda la mejor solución encontrada
      mejor_distancia = distancia_vecina

    else:
      print("En la iteracion ", iteracion, ", la mejor solución encontrada es:" , mejor_solucion)
      print("Distancia     :" , mejor_distancia)
      return mejor_solucion

    solucion_referencia = vecina


sol = busqueda_local(problem )

En la iteracion  29 , la mejor solución encontrada es: [0, 14, 16, 15, 20, 33, 34, 38, 22, 39, 32, 7, 37, 17, 36, 35, 31, 27, 2, 6, 13, 19, 1, 28, 30, 29, 8, 9, 21, 24, 40, 23, 41, 25, 10, 26, 5, 18, 12, 11, 4, 3]
Distancia     : 1858


### Modificar la estructura de entornos búsqueda en entornos variables
* Búsqueda por Entornos Variables para Planificación Logística:

In [11]:
# Crear generadores de vecindad optimizados
def genera_vecina_2opt(solucion, problem):
    mejor_solucion = solucion[:]
    mejor_distancia = distancia_total(solucion, problem)

    for i in range(1, len(solucion) - 2):
        for j in range(i + 1, len(solucion) - 1):
            vecina = solucion[:i] + solucion[i:j+1][::-1] + solucion[j+1:]  # Intercambio eficiente
            distancia_vecina = distancia_total(vecina, problem)
            if distancia_vecina < mejor_distancia:
                mejor_distancia = distancia_vecina
                mejor_solucion = vecina

    return mejor_solucion

def genera_vecina_3opt(solucion, problem):
    mejor_solucion = solucion[:]
    mejor_distancia = distancia_total(solucion, problem)
    n = len(solucion)

    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            for k in range(j + 1, n):
                # Generar solo las permutaciones más prometedoras
                vecinas = [
                    solucion[:i+1] + solucion[j:k+1] + solucion[i+1:j] + solucion[k+1:],
                    solucion[:i+1] + solucion[j:k+1][::-1] + solucion[i+1:j] + solucion[k+1:]
                ]

                for s in vecinas:
                    d = distancia_total(s, problem)
                    if d < mejor_distancia:
                        mejor_distancia = d
                        mejor_solucion = s

    return mejor_solucion

# Hacer una búsqueda en Entornos Variables optimizada
def busqueda_entornos_variables(problem):
    solucion = crear_solucion(Nodos)
    mejor_distancia = distancia_total(solucion, problem)
    iteracion = 0
    k = 1

    while k <= 2:
        iteracion += 1
        vecina = genera_vecina_2opt(solucion, problem) if k == 1 else genera_vecina_3opt(solucion, problem)
        distancia_vecina = distancia_total(vecina, problem)

        if distancia_vecina < mejor_distancia:
            mejor_distancia = distancia_vecina
            solucion = vecina[:]
            k = 1  # Reiniciar vecindad
        else:
            k += 1  # Cambiar a la siguiente vecindad

    print(f"Mejor solución encontrada en {iteracion} iteraciones: {solucion}")
    print(f"Distancia: {mejor_distancia}")
    return solucion

# Ejecución del algoritmo optimizado
mejor_solucion = busqueda_entornos_variables(problem)



Mejor solución encontrada en 42 iteraciones: [0, 1, 6, 5, 13, 19, 14, 16, 15, 37, 7, 17, 31, 36, 35, 20, 33, 34, 32, 27, 2, 28, 29, 30, 38, 22, 39, 24, 40, 21, 9, 23, 41, 8, 10, 25, 11, 12, 18, 26, 4, 3]
Distancia: 1312


### Busqueda Tabú
La Búsqueda Tabú es una técnica de búsqueda local que utiliza una lista tabú para evitar ciclos y se permite explorar soluciones peores temporalmente. La lista tabú guarda movimientos recientes que no deben ser deshechos inmediatamente.

In [12]:
# Búsqueda Tabú Optimizada
def busqueda_tabu(problem, max_iter, lista_tabu_tam):
    """
    Implementa la metaheurística de Búsqueda Tabú para encontrar una solución óptima en un problema de optimización combinatoria.

    Parámetros:
    - problem: instancia del problema a resolver.
    - max_iter: número máximo de iteraciones.
    - lista_tabu_tam: tamaño de la lista tabú para evitar ciclos en la exploración.

    Retorna:
    - mejor_solucion: la mejor solución encontrada durante la búsqueda.
    """

    # Se inicializa una solución aleatoria y se almacena como la mejor encontrada hasta el momento
    solucion_actual = crear_solucion(Nodos)
    mejor_solucion = solucion_actual[:]
    mejor_distancia = distancia_total(solucion_actual, problem)
    lista_tabu = set()  # Se usa un conjunto para mejorar la eficiencia en la búsqueda

    for iteracion in range(max_iter):
        vecina = genera_vecina_2opt(solucion_actual, problem)  # Se pasa 'problem' correctamente
        distancia_actual = distancia_total(vecina, problem)  # Se calcula una única vez

        # Se verifica que la solución vecina no esté en la lista tabú
        if tuple(vecina) not in lista_tabu:
            solucion_actual = vecina

            # Si la nueva solución es mejor, se actualiza la mejor conocida
            if distancia_actual < mejor_distancia:
                mejor_solucion = solucion_actual[:]
                mejor_distancia = distancia_actual

            # Se actualiza la lista tabú
            lista_tabu.add(tuple(vecina))
            if len(lista_tabu) > lista_tabu_tam:
                lista_tabu.pop()  # Se elimina el elemento más antiguo para mantener el tamaño

    # Se imprimen los resultados obtenidos
    print(f"Mejor solución encontrada en {iteracion + 1} iteraciones: {mejor_solucion}")
    print(f"Distancia: {mejor_distancia}")
    return mejor_solucion




In [13]:
mejor_solucion_tabu = busqueda_tabu(problem, 100, 10)


Mejor solución encontrada en 100 iteraciones: [0, 32, 34, 33, 20, 35, 36, 31, 17, 1, 7, 37, 15, 16, 14, 19, 13, 5, 26, 18, 12, 11, 25, 10, 41, 8, 9, 23, 40, 24, 21, 39, 22, 38, 30, 29, 28, 2, 27, 3, 4, 6]
Distancia: 1346


In [14]:
mejor_solucion_tabu = busqueda_tabu(problem, 50, 5)


Mejor solución encontrada en 50 iteraciones: [0, 27, 2, 3, 4, 6, 1, 7, 31, 35, 36, 17, 37, 15, 16, 14, 19, 13, 5, 26, 18, 12, 11, 25, 10, 41, 23, 9, 8, 29, 30, 28, 32, 20, 33, 34, 38, 22, 39, 21, 24, 40]
Distancia: 1447


### Recocido Simulado
El Recocido Simulado es un método de optimización basado en principios de mecánica estadística, el cual permite la transición hacia soluciones de menor calidad con una probabilidad que decrece a medida que avanza el proceso de enfriamiento. Esta probabilidad de aceptación de soluciones subóptimas está determinada por una función de temperatura, la cual se reduce progresivamente con el tiempo, favoreciendo la exploración en las etapas iniciales y promoviendo la convergencia hacia soluciones óptimas en las fases finales.

In [15]:
import math
import random

def probabilidad_aceptacion(delta_e, temperatura):
    """
    Calcula la probabilidad de aceptar una solución peor en el Recocido Simulado.

    Parámetros:
    - delta_e: diferencia entre la calidad de la nueva solución y la actual.
    - temperatura: temperatura actual en el proceso de enfriamiento.

    Retorna:
    - Probabilidad de aceptación, que varía entre 0 y 1.
    """
    return 1.0 if delta_e < 0 else math.exp(-delta_e / temperatura)

# Algoritmo de Recocido Simulado Corregido
def recocido_simulado(problem, temp_inicial, temp_final, alpha, max_iter):
    """
    Implementa el algoritmo de Recocido Simulado para optimización combinatoria.

    Parámetros:
    - problem: instancia del problema a resolver.
    - temp_inicial: temperatura inicial del proceso de enfriamiento.
    - temp_final: temperatura final del proceso.
    - alpha: factor de enfriamiento, debe ser menor que 1.
    - max_iter: número máximo de iteraciones en cada nivel de temperatura.

    Retorna:
    - mejor_solucion: la mejor solución encontrada durante la ejecución.
    """

    # Se genera una solución inicial aleatoria y se establece como la mejor encontrada
    solucion_actual = crear_solucion(Nodos)
    mejor_solucion = solucion_actual[:]
    mejor_distancia = distancia_total(solucion_actual, problem)
    temperatura = temp_inicial

    while temperatura > temp_final:
        for _ in range(max_iter):
            # Se genera una solución vecina con el argumento 'problem' correctamente pasado
            vecina = genera_vecina_2opt(solucion_actual, problem)
            distancia_vecina = distancia_total(vecina, problem)
            delta_e = distancia_vecina - mejor_distancia  # Cambio en calidad de la solución

            # Se decide si se acepta la nueva solución con base en la probabilidad de aceptación
            if probabilidad_aceptacion(delta_e, temperatura) > random.random():
                solucion_actual = vecina[:]
                if distancia_vecina < mejor_distancia:
                    mejor_solucion = solucion_actual[:]
                    mejor_distancia = distancia_vecina

        # Se reduce la temperatura aplicando el factor de enfriamiento
        temperatura *= alpha

    # Se imprimen los resultados finales del algoritmo
    print(f"Mejor solución encontrada: {mejor_solucion}")
    print(f"Distancia: {mejor_distancia}")
    return mejor_solucion

# Se ejecuta el algoritmo de Recocido Simulado
mejor_solucion_recocido = recocido_simulado(problem, 5, 1, 0.99, 1)



Mejor solución encontrada: [0, 7, 1, 6, 4, 3, 27, 2, 8, 9, 41, 23, 21, 40, 24, 39, 22, 38, 30, 29, 28, 32, 34, 33, 20, 35, 36, 31, 17, 37, 15, 16, 14, 19, 13, 5, 26, 18, 12, 10, 25, 11]
Distancia: 1445


### Algoritmos geneticos

In [16]:
# Función para generar una población inicial de soluciones de tamaño N
def generar_poblacion(Nodos, N):
    poblacion = []
    for _ in range(N):
        solucion = crear_solucion(Nodos)
        poblacion.append(solucion)
    return poblacion

# Ejemplo de uso
Nodos = list(problem.get_nodes())  # Asegúrate de tener la lista de nodos del problema
tamano_poblacion = 15  # Define el tamaño de la población
poblacion_inicial = generar_poblacion(Nodos, tamano_poblacion)

# Imprime la población inicial
for idx, solucion in enumerate(poblacion_inicial):
    print(f"Solución {idx+1}: {solucion}")


Solución 1: [0, 22, 34, 11, 26, 7, 32, 25, 39, 16, 4, 29, 40, 20, 2, 3, 9, 17, 19, 18, 23, 37, 35, 21, 6, 8, 31, 5, 33, 38, 36, 14, 30, 41, 27, 1, 28, 13, 15, 10, 12, 24]
Solución 2: [0, 21, 7, 31, 8, 3, 20, 30, 2, 24, 13, 12, 10, 34, 6, 22, 41, 19, 39, 29, 28, 4, 32, 37, 17, 9, 36, 16, 15, 11, 40, 14, 38, 33, 25, 1, 23, 27, 5, 26, 35, 18]
Solución 3: [0, 1, 38, 11, 20, 2, 16, 21, 18, 12, 22, 6, 28, 15, 30, 4, 25, 19, 36, 8, 31, 23, 27, 24, 10, 37, 40, 32, 13, 5, 41, 9, 35, 26, 17, 34, 39, 14, 33, 3, 29, 7]
Solución 4: [0, 32, 18, 20, 17, 40, 41, 28, 6, 4, 23, 39, 5, 14, 37, 33, 29, 22, 38, 13, 11, 35, 24, 36, 9, 1, 3, 8, 12, 21, 10, 34, 27, 2, 31, 19, 16, 25, 15, 26, 7, 30]
Solución 5: [0, 2, 21, 26, 27, 37, 7, 10, 22, 38, 32, 5, 33, 18, 6, 15, 12, 14, 31, 30, 24, 19, 36, 35, 9, 8, 41, 11, 3, 1, 16, 39, 17, 29, 40, 34, 25, 13, 4, 28, 23, 20]
Solución 6: [0, 39, 7, 16, 35, 22, 6, 40, 26, 8, 19, 5, 33, 36, 9, 25, 3, 27, 31, 32, 11, 1, 10, 13, 28, 12, 30, 41, 23, 15, 14, 37, 21, 29, 2, 2

Evaluar poblacion

In [17]:
#Evalua la población y devuelve el mejor individuo
def Evaluar_Poblacion(poblacion, problem):
    mejor_solucion = None
    mejor_distancia = float('inf')

    for solucion in poblacion:
        distancia = distancia_total(solucion, problem)
        if distancia < mejor_distancia:
            mejor_solucion = solucion
            mejor_distancia = distancia

    return mejor_solucion, mejor_distancia

# Ejemplo de uso
poblacion_inicial = generar_poblacion(Nodos, 10)
mejor_solucion, mejor_distancia = Evaluar_Poblacion(poblacion_inicial, problem)

print("Mejor solución:", mejor_solucion)
print("Distancia de la mejor solución:", mejor_distancia)


Mejor solución: [0, 14, 20, 37, 16, 29, 26, 18, 28, 41, 30, 25, 23, 11, 12, 36, 13, 2, 3, 7, 35, 6, 33, 5, 1, 22, 9, 27, 10, 17, 34, 32, 8, 19, 38, 31, 4, 21, 39, 24, 40, 15]
Distancia de la mejor solución: 4202


Función de cruce


In [18]:
import random

def cruzar_individuos(padre1, padre2):
    # Cruce de un punto
    punto_cruce = random.randint(1, len(padre1) - 2)

    hijo1 = padre1[:punto_cruce] + [gen for gen in padre2 if gen not in padre1[:punto_cruce]]
    hijo2 = padre2[:punto_cruce] + [gen for gen in padre1 if gen not in padre2[:punto_cruce]]

    return hijo1, hijo2

def mutar_individuo(individuo, tasa_mutacion):
    for i in range(len(individuo)):
        if random.random() < tasa_mutacion:
            j = random.randint(0, len(individuo) - 1)
            individuo[i], individuo[j] = individuo[j], individuo[i]  # Intercambiar dos genes
    return individuo

def Cruzar(poblacion, tasa_mutacion, problem):
    nueva_poblacion = []
    poblacion_ampliada = []

    # Mezclar la población para emparejar aleatoriamente
    random.shuffle(poblacion)

    # Realizar el cruce para cada par de individuos en la población
    for i in range(0, len(poblacion) - 1, 2):
        padre1 = poblacion[i]
        padre2 = poblacion[i + 1]

        # Generar dos hijos a partir de los dos padres
        hijo1, hijo2 = cruzar_individuos(padre1, padre2)

        # Aplicar mutación a los hijos
        hijo1 = mutar_individuo(hijo1, tasa_mutacion)
        hijo2 = mutar_individuo(hijo2, tasa_mutacion)

        # Agregar los hijos a la nueva población
        nueva_poblacion.extend([hijo1, hijo2])

    # Combinar la población original y la nueva población generada
    poblacion_ampliada = poblacion + nueva_poblacion

    return poblacion_ampliada

# Ejemplo de uso
poblacion_inicial = generar_poblacion(Nodos, 10)
tasa_mutacion = 0.05  # Porcentaje de mutación
poblacion_cruzada = Cruzar(poblacion_inicial, tasa_mutacion, problem)

print("Población inicial:", poblacion_inicial)
print("Población después del cruce y mutación:", poblacion_cruzada)


Población inicial: [[0, 27, 25, 41, 30, 28, 15, 7, 31, 24, 5, 32, 38, 2, 21, 6, 16, 1, 20, 19, 11, 4, 3, 29, 40, 8, 26, 36, 35, 22, 17, 37, 18, 33, 34, 13, 23, 39, 14, 12, 9, 10], [0, 8, 11, 15, 32, 29, 12, 27, 23, 20, 31, 41, 4, 21, 36, 16, 25, 2, 7, 26, 5, 17, 19, 18, 24, 40, 38, 37, 33, 9, 14, 30, 28, 39, 1, 3, 6, 35, 13, 10, 34, 22], [0, 3, 34, 17, 13, 16, 11, 37, 27, 1, 33, 41, 22, 9, 19, 36, 26, 23, 6, 21, 28, 5, 8, 39, 15, 14, 18, 20, 32, 30, 40, 24, 29, 10, 2, 4, 25, 12, 31, 38, 7, 35], [0, 39, 29, 22, 30, 10, 8, 16, 26, 12, 36, 33, 13, 34, 25, 3, 32, 37, 38, 19, 9, 1, 17, 2, 14, 27, 20, 23, 41, 6, 5, 18, 31, 7, 21, 11, 28, 40, 35, 24, 15, 4], [0, 8, 40, 5, 37, 24, 34, 30, 28, 21, 14, 3, 16, 29, 11, 22, 33, 15, 31, 25, 26, 2, 12, 35, 1, 6, 10, 13, 36, 27, 9, 19, 20, 18, 32, 23, 4, 39, 7, 17, 38, 41], [0, 19, 40, 8, 17, 12, 5, 11, 22, 21, 20, 35, 15, 14, 9, 38, 13, 29, 33, 39, 16, 25, 23, 3, 1, 24, 36, 37, 31, 27, 18, 7, 4, 2, 30, 28, 32, 41, 10, 26, 6, 34], [0, 39, 1, 27, 10, 3

Descendencia

In [19]:
import random

def Descendencia(padres, problem, mutacion):
    padre1, padre2 = padres
    punto_corte = random.randint(1, len(padre1) - 2)

    # Crear hijos utilizando el corte de un punto
    hijo1 = padre1[:punto_corte] + [nodo for nodo in padre2 if nodo not in padre1[:punto_corte]]
    hijo2 = padre2[:punto_corte] + [nodo for nodo in padre1 if nodo not in padre2[:punto_corte]]

    # Aplicar mutación a los hijos
    hijo1 = aplicar_mutacion(hijo1, mutacion)
    hijo2 = aplicar_mutacion(hijo2, mutacion)

    return hijo1, hijo2

def aplicar_mutacion(solucion, tasa_mutacion):
    for i in range(len(solucion)):
        if random.random() < tasa_mutacion:
            j = random.randint(0, len(solucion) - 1)
            # Intercambiar dos genes para mutar
            solucion[i], solucion[j] = solucion[j], solucion[i]
    return solucion


In [20]:
# Supongamos que tienes una población y seleccionas dos padres
padres = (poblacion_inicial[0], poblacion_inicial[1])
tasa_mutacion = 0.05  # 5% de probabilidad de mutación

# Generar descendencia
hijo1, hijo2 = Descendencia(padres, problem, tasa_mutacion)

print("Hijo 1:", hijo1)
print("Hijo 2:", hijo2)


Hijo 1: [0, 27, 25, 41, 30, 28, 15, 7, 34, 24, 5, 32, 38, 2, 21, 6, 16, 1, 20, 19, 11, 4, 3, 29, 40, 8, 26, 36, 35, 22, 17, 37, 18, 12, 23, 33, 9, 14, 39, 13, 10, 31]
Hijo 2: [0, 8, 11, 15, 32, 29, 12, 27, 23, 20, 31, 41, 4, 21, 36, 16, 25, 2, 7, 26, 5, 17, 19, 18, 24, 40, 38, 13, 33, 9, 14, 30, 28, 6, 1, 3, 35, 22, 34, 37, 39, 10]


Factibilizar

In [21]:
def Factibilizar(solucion, problem):
    Nodos = list(problem.get_nodes())
    solucion_factible = []
    nodos_presentes = set()

    # Añadir solo nodos únicos a la solución factible
    for nodo in solucion:
        if nodo not in nodos_presentes:
            solucion_factible.append(nodo)
            nodos_presentes.add(nodo)

    # Añadir nodos que faltan para completar la solución
    for nodo in Nodos:
        if nodo not in nodos_presentes:
            solucion_factible.append(nodo)

    return solucion_factible

# Ejemplo de uso
solucion = [1, 2, 3, 4, 1, 5, 6]  # Suponiendo una solución con repeticiones o nodos faltantes
solucion_factible = Factibilizar(solucion, problem)
print("Solución original:", solucion)
print("Solución factibilizada:", solucion_factible)


Solución original: [1, 2, 3, 4, 1, 5, 6]
Solución factibilizada: [1, 2, 3, 4, 5, 6, 0, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]


Mutar:

In [22]:
import random

def Mutar(solucion, mutacion):
    """
    Mutar una solución intercambiando dos nodos con una probabilidad definida por la tasa de mutación.
    Args:
    - solucion: Lista de nodos representando la solución actual.
    - mutacion: Tasa de mutación (probabilidad de mutar).
    """
    # Aplicar mutación solo con una probabilidad igual a la tasa de mutación
    if random.random() < mutacion:
        # Elegir dos índices aleatorios distintos para el intercambio
        i, j = random.sample(range(len(solucion)), 2)
        # Realizar el intercambio
        solucion[i], solucion[j] = solucion[j], solucion[i]
    return solucion

# Ejemplo de uso
solucion = [1, 2, 3, 4, 5]
tasa_mutacion = 0.1  # 10% de probabilidad de mutación
solucion_mutada = Mutar(solucion, tasa_mutacion)
print("Solución original:", solucion)
print("Solución mutada:", solucion_mutada)


Solución original: [1, 2, 3, 4, 5]
Solución mutada: [1, 2, 3, 4, 5]


Seleccionar

In [23]:
import random

def Seleccionar(problem, poblacion, N, elitismo):
    """
    Selecciona individuos de una población para formar la siguiente generación.
    Args:
    - problem: instancia del problema para evaluar la aptitud.
    - poblacion: lista de individuos de la generación actual.
    - N: tamaño deseado de la población.
    - elitismo: porcentaje de la población que se selecciona basado en el mejor rendimiento.
    """
    # Calcular la aptitud para cada individuo
    aptitudes = [(individuo, distancia_total(individuo, problem)) for individuo in poblacion]

    # Ordenar individuos por su aptitud (de menor a mayor distancia)
    aptitudes.sort(key=lambda x: x[1])

    # Seleccionar el top de individuos como élite
    num_elite = int(elitismo * N)
    nueva_poblacion = [individuo for individuo, _ in aptitudes[:num_elite]]

    # Seleccionar el resto de individuos mediante la selección de ruleta
    # Convertir aptitudes a probabilidades inversas (fitness menor es mejor)
    total_fitness = sum([1 / aptitud for _, aptitud in aptitudes[num_elite:]])
    probabilidades = [(1 / aptitud) / total_fitness for _, aptitud in aptitudes[num_elite:]]

    # Elegir los individuos restantes usando selección de ruleta
    poblacion_restante = random.choices(
        population=[individuo for individuo, _ in aptitudes[num_elite:]],
        weights=probabilidades,
        k=N - num_elite
    )

    # Añadir los individuos seleccionados por ruleta a la nueva población
    nueva_poblacion.extend(poblacion_restante)

    return nueva_poblacion

# Ejemplo de uso
poblacion_inicial = generar_poblacion(Nodos, 100)
poblacion_seleccionada = Seleccionar(problem, poblacion_inicial, 100, 0.2)


In [24]:
#Funcion principal del algoritmo genetico
#######################################################3
def algoritmo_genetico(problem=problem,N=100,mutacion=.15,elitismo=.1,generaciones=100):
  # problem = datos del problema
  # N = Tamaño de la población
  # mutacion = probabilidad de una mutación
  # elitismo = porcion de la mejor poblacion a mantener
  # generaciones = nº de generaciones a generar para finalizar

  #Genera la poblacion inicial
  Nodos = list(problem.get_nodes())
  poblacion = generar_poblacion(Nodos,N)

  #Inicializamos valores para la mejor solucion
  (mejor_solucion, mejor_distancia) = Evaluar_Poblacion(poblacion, problem)


  #Condicion de parada
  parar = False
  n=0
  #Inciamos el cliclo de generaciones
  while(parar == False) :

    #Cruce de la poblacion(incluye mutación)
    poblacion = Cruzar(poblacion, mutacion, problem)

    #Seleccionamos la población
    poblacion = Seleccionar(problem,poblacion, N, elitismo)

    #Evaluamos la nueva población
    (mejor_solucion, mejor_distancia) = Evaluar_Poblacion(poblacion, problem)

    print("Generacion #", n, "\nLa mejor solución es:" , mejor_solucion, "\ncon distancia " , mejor_distancia, "\n")

    #Numero de generaciones. Criterio de parada
    if n==generaciones:
      parar = True
    n +=1

  return mejor_solucion


sol = algoritmo_genetico(problem=problem,N=500,mutacion=.3,elitismo=.40,generaciones=100)

Generacion # 0 
La mejor solución es: [0, 13, 3, 23, 12, 25, 22, 16, 35, 6, 15, 33, 37, 14, 28, 27, 19, 5, 7, 24, 32, 26, 4, 10, 41, 9, 2, 1, 18, 11, 36, 17, 29, 34, 31, 20, 38, 30, 8, 40, 21, 39] 
con distancia  3817 

Generacion # 1 
La mejor solución es: [0, 13, 3, 23, 12, 25, 22, 16, 35, 6, 15, 33, 37, 14, 28, 27, 19, 5, 7, 24, 32, 26, 4, 10, 41, 9, 2, 1, 18, 11, 36, 17, 29, 34, 31, 20, 38, 30, 8, 40, 21, 39] 
con distancia  3817 

Generacion # 2 
La mejor solución es: [10, 3, 4, 33, 20, 35, 36, 7, 37, 1, 24, 28, 31, 32, 29, 21, 39, 11, 27, 26, 23, 6, 12, 15, 19, 34, 5, 17, 9, 2, 30, 0, 25, 8, 38, 22, 18, 16, 14, 13, 41, 40] 
con distancia  3755 

Generacion # 3 
La mejor solución es: [10, 3, 4, 33, 20, 35, 36, 7, 37, 1, 24, 28, 31, 32, 29, 21, 39, 11, 27, 26, 23, 6, 12, 15, 19, 34, 5, 17, 9, 2, 30, 0, 25, 8, 38, 22, 18, 16, 14, 13, 41, 40] 
con distancia  3755 

Generacion # 4 
La mejor solución es: [10, 3, 4, 33, 20, 35, 36, 7, 37, 1, 24, 28, 31, 32, 29, 21, 39, 11, 27, 26, 23, 6

# Algoritmo de colonia de hormigas

La función Add_Nodo selecciona al azar un nodo con probabilidad uniforme.

In [25]:
def Add_Nodo(problem, H ,T ) :
  #Mejora:Establecer una funcion de probabilidad para
  # añadir un nuevo nodo dependiendo de los nodos mas cercanos y de las feromonas depositadas
  Nodos = list(problem.get_nodes())
  return random.choice(   list(set(range(1,len(Nodos))) - set(H) )  )


def Incrementa_Feromona(problem, T, H ) :
  #Incrementa segun la calidad de la solución. Añadir una cantidad inversamente proporcional a la distancia total
  for i in range(len(H)-1):
    T[H[i]][H[i+1]] += 1000/distancia_total(H, problem)
  return T

def Evaporar_Feromonas(T ):
  #Evapora 0.3 el valor de la feromona, sin que baje de 1
  #Mejora:Podemos elegir diferentes funciones de evaporación dependiendo de la cantidad actual y de la suma total de feromonas depositadas,...
  T = [[ max(T[i][j] - 0.3 , 1) for i in range(len(Nodos)) ] for j in range(len(Nodos))]
  return T

In [26]:
def hormigas(problem, N) :
  #problem = datos del problema
  #N = Número de agentes(hormigas)

  #Nodos
  Nodos = list(problem.get_nodes())
  #Aristas
  Aristas = list(problem.get_edges())

  #Inicializa las aristas con una cantidad inicial de feromonas:1
  #Mejora: inicializar con valores diferentes dependiendo diferentes criterios
  T = [[ 1 for _ in range(len(Nodos)) ] for _ in range(len(Nodos))]

  #Se generan los agentes(hormigas) que serán estructuras de caminos desde 0
  Hormiga = [[0] for _ in range(N)]

  #Recorre cada agente construyendo la solución
  for h in range(N) :
    #Para cada agente se construye un camino
    for i in range(len(Nodos)-1) :

      #Elige el siguiente nodo
      Nuevo_Nodo = Add_Nodo(problem, Hormiga[h] ,T )
      Hormiga[h].append(Nuevo_Nodo)

    #Incrementa feromonas en esa arista
    T = Incrementa_Feromona(problem, T, Hormiga[h] )
    #print("Feromonas(1)", T)

    #Evapora Feromonas
    T = Evaporar_Feromonas(T)
    #print("Feromonas(2)", T)

    #Seleccionamos el mejor agente
  mejor_solucion = []
  mejor_distancia = 10e100
  for h in range(N) :
    distancia_actual = distancia_total(Hormiga[h], problem)
    if distancia_actual < mejor_distancia:
      mejor_solucion = Hormiga[h]
      mejor_distancia =distancia_actual


  print(mejor_solucion)
  print(mejor_distancia)


hormigas(problem, 1000)

[0, 34, 30, 28, 16, 41, 29, 27, 21, 5, 19, 26, 13, 20, 22, 4, 32, 7, 25, 24, 9, 23, 40, 1, 8, 2, 38, 39, 33, 6, 15, 10, 35, 36, 14, 37, 3, 18, 11, 12, 17, 31]
3934
